# PCA reconstruction anomaly-detection baseline

This mirrors TEP notebook 3.1. PCA is fit on the three endogenous process variables from healthy synthetic training samples, retaining 95% of the variance where possible. The component count is capped at two so the squared prediction error (SPE) remains a meaningful reconstruction score. Faulty validation runs select the alarm settings; all faulty and healthy held-out runs are then tested without refitting the threshold.

In [1]:
from pathlib import Path
import sys


def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from IPython.display import display
from carl.causalai_baselines import run_baseline_comparison

baseline_results = run_baseline_comparison("pca", PROJECT_ROOT)
print("PCA baseline fit, validation-tuned, and evaluated on held-out test runs.")

PCA baseline fit, validation-tuned, and evaluated on held-out test runs.


## 2. Validation and hyperparameter selection

For each moving-average window, the threshold is the maximum smoothed SPE in the healthy first phase of faulty validation runs. Window and consecutive alarms maximize trajectory-phase F1, with median detection delay breaking ties.

In [2]:
display(baseline_results["validation_grid"].sort_values(
    ["Trajectory-Phase F1", "Median Anomaly Detection Delay"],
    ascending=[False, True],
).reset_index(drop=True))
display(baseline_results["selected_settings"])

,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Detected Runs,Total Faulty Runs,Median Anomaly Detection Delay
0,8,5,1.882515,0.377472,0.230400,0.0,0.374512,0.880000,0.760000,0.0,0.863636,171,225,49.0
1,8,6,1.882515,0.370031,0.221989,0.0,0.363324,0.868889,0.737778,0.0,0.849105,166,225,50.0
2,7,5,2.094473,0.365298,0.215350,0.0,0.354384,0.864444,0.728889,0.0,0.843188,164,225,49.0
3,8,7,1.882515,0.362927,0.214011,0.0,0.352569,0.862222,0.724444,0.0,0.840206,163,225,51.0
4,7,6,2.094473,0.357935,0.207050,0.0,0.343068,0.857778,0.715556,0.0,0.834197,161,225,50.0
5,8,8,1.882515,0.356272,0.206606,0.0,0.342457,0.851111,0.702222,0.0,0.825065,158,225,53.0
6,7,7,2.094473,0.351030,0.199333,0.0,0.332407,0.848889,0.697778,0.0,0.821990,157,225,53.0
7,6,5,2.360392,0.352504,0.199533,0.0,0.332685,0.844444,0.688889,0.0,0.815789,155,225,49.0
8,6,6,2.360392,0.345367,0.191528,0.0,0.321483,0.842222,0.684444,0.0,0.812665,154,225,54.0
9,5,5,2.721061,0.339216,0.183106,0.0,0.309534,0.831111,0.662222,0.0,0.796791,149,225,53.0


,Baseline,Max Lag,Window Size,Consecutive Alarms,Threshold,PCA Components
0,pca,2,8,5,1.882515,2


## 3. Test on faulty time series

The validation-selected settings are fixed for the full faulty test set.

In [3]:
display(baseline_results["faulty_test_metrics"])
display(baseline_results["faulty_test_by_type"])

,Baseline,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Detected Runs,Total Faulty Runs,Median Anomaly Detection Delay
0,pca,2,8,5,1.882515,0.393282,0.249944,0.0,0.399929,0.9,0.8,0.0,0.888889,180,225,56.0


,Fault Type,Runs,Detected_Runs,Run_Detection_Rate,Median_Detection_Delay
0,hydraulic_coupling_shift,75,35,0.466667,402.0
1,reaction_coupling_shift,75,74,0.986667,41.0
2,thermal_coupling_shift,75,71,0.946667,41.0


## 4. Test on fault-free time series

The same fixed threshold is evaluated on all healthy held-out runs. The comparison table is updated as each of notebooks 3.0, 3.1, and 3.2 is run.

In [4]:
display(baseline_results["healthy_test_metrics"])
display(baseline_results["comparison"])

,Baseline,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase FPR,False Alarm Runs,Total Healthy Runs
0,pca,2,8,5,1.882515,1.0,0.0,0.0,0.0,0.0,0,75


,Baseline,Max Lag,Window Size,Consecutive Alarms,Threshold,PCA Components,Faulty Test Sample F1,Faulty Test Trajectory-Phase F1,Faulty Test Trajectory-Phase TPR,Faulty Test Median Detection Delay,Healthy Test Sample FPR,Healthy Test Trajectory-Phase FPR
0,full_var,2,8,5,2.983853,NaN,0.592920,0.910412,0.835556,31.5,0.0,0.0
2,pca,2,8,5,1.882515,2.0,0.399929,0.888889,0.800000,56.0,0.0,0.0
1,self_history,2,7,5,3.407215,NaN,0.483112,0.728814,0.573333,32.0,0.0,0.0
